## Часть 3

Источник данных: https://github.com/rfordatascience/tidytuesday/tree/main/data/2026/2026-03-10

In an online quiz, created as an independent project by Adam Kucharski, over 5,000 participants compared pairs of probability phrases (e.g. “Which conveys a higher probability: Likely or Probable?”) and assigned numerical values (0–100%) to each of 19 phrases. The resulting data can be used to analyse how people interpret common probability phrases.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

Абсолютные оценки вероятностей, обозначаемых словами

In [2]:
abs_judgements_df = pd.read_csv("data/absolute_judgements.csv")
abs_judgements_df.sample(3)

,response_id,term,probability,order
72821,1345,Chances are Slight,10,14
36086,3278,Unlikely,30,6
41311,3003,Almost Certain,95,6


Попарные сравнения вероятностей, обозначаемых словами

In [3]:
pairwise_df = pd.read_csv("data/pairwise_comparisons.csv")
pairwise_df.sample(3)

,response_id,pair_id,term1,term2,selected
31794,1998,5,Will Happen,May Happen,Will Happen
45675,610,6,May Happen,Will Happen,Will Happen
49761,201,2,Realistic Possibility,Better than Even,Better than Even


Данные о респондентах

In [4]:
resp_df = pd.read_csv("data/respondent_metadata.csv")
resp_df.sample(5)

,response_id,timestamp,age_band,english_background,education_level,country_of_residence
1442,3735,2026-01,45-54,English is my first language,Bachelor,United Kingdom
1703,3474,2026-01,NaN,NaN,NaN,NaN
4014,1163,2026-01,35-44,English is not my first language but I am fluent,Postgraduate,United States
4762,415,2026-01,35-44,English is not my first language but I am fluent,Postgraduate,France
3146,2031,2026-01,18-24,English is my first language,Postgraduate,United Kingdom


### Корреляция

### Согласованность мнений экспертов

In [5]:
def kendall_concordance(df, feature_col="term", value_col="probability", expert_col="response_id"):
    """
    Вычисление коэффициента конкордации Кендалла.
    """

    m = df[expert_col].nunique() # кол-во экспертов
    n = df.groupby(expert_col)[value_col].count().values[0] # кол-во оцениваемых величин

    df["rank"] = df.groupby(expert_col)[value_col].rank(method="average", ascending=False).astype(int) # ранги для оценок каждого эксперта 
    rank_sum = df.groupby(feature_col)["rank"].sum() # сумма рангов для каждого слова
    rank_mean = rank_sum.sum() / n # средняя сумма рангов по всем словам
    delta_rank = rank_sum - rank_mean # отклонение суммы рангов каждого слова
    delta_rank_sq = np.square(delta_rank) # квадрат отклонения
    S = delta_rank_sq.sum()
    return 12 * S / (m ** 2 * (np.pow(n, 3) - n))

Вычисление ранговой согласованности мнений респондентов относительно значений вероятностей, определяемых представленными словами.

H0: согласованность высокая (W >= 0.8)

H1: согласованность низкая (W < 0.8)

In [8]:
W = round(kendall_concordance(abs_judgements_df), 3)
print(f"Коэффициент конкордации для оценок вероятностей, описываемых словами W = {W}")

Коэффициент конкордации для оценок вероятностей, описываемых словами W = 0.875


W > 0.8, поэтому H0 принимается - согласованность экспертов достаточно высокая, что свидетельствует о схожем представлении о различии синонимичных слов среди людей с разным уровнем знания английского.

Проверка значимости W с помощью $ \chi^2 $-критерия при уровне значимости $ \alpha=0.05 $

H0: согласованность случайная ($ \chi^2_{расч.} < \chi^2_{табл.} $)

H1: согласованность значима ($ \chi^2_{расч.} \geqslant \chi^2_{табл.} $)

In [11]:
m = abs_judgements_df["response_id"].nunique() # кол-во экспертов
n = abs_judgements_df.groupby("response_id")["probability"].count().values[0] # кол-во оцениваемых величин
chi2 = m * (n - 1) * W # хи-квадрат
print(f"""Коэффициент хи-квадрат = {chi2}
          Кол-во экспертов = {m}
          Число степеней свободы = {n}
          Уровень значимости alpha = 0.05""")

Коэффициент хи-квадрат = 81490.5
          Кол-во экспертов = 5174
          Число степеней свободы = 19
          Уровень значимости alpha = 0.05


Для n = 19 и $\alpha$ = 0.05 табличное значение $\chi^2$ = 30.1. Расчетное значение сильно превышает табличное, поэтому коэффициент конкордации статистически значим. H0 отвергается.